In [1]:
# Imports
import duckdb
import pandas as pd

# 2. Spotlight Prep
This notebook... 

## Write relevant .csv files to get Spotlight predictions

In [3]:
# Load data with old Spotlight predictions
fullmatch_df = pd.read_csv('../data/SpotlightWeightSource_0102_0505_fullmatch_corrected.csv')
fullmatch_df.head()

# Load sample of 46 source biographies
napo_sample = pd.read_csv('../data/napo_sample.csv')

In [8]:
con = duckdb.connect('../data/out/napo_sources.db')

source_biographies = con.execute("""
    SELECT 'Q' || wp.wikidata_id, wp.language_code, e.extract
    FROM wiki_page wp
    JOIN extract e ON e.wiki_page_id = wp.wiki_page_id
""").fetchall()

con.close()

source_biographies = pd.DataFrame(source_biographies, columns=["wikidata_id", "language_code", "extract"])
source_biographies.shape

(46, 3)

In [9]:
napoleon_links = napo_sample[["source_wikidata_id", "target_wikidata_id", "language_code"]]
napoleon_links.shape

(46, 3)

In [ ]:
# Write csv files in folder for HPC
source_biographies.to_csv('../hpc_jobs/hpc_spotlight/data/source_biographies.csv', index=False)
napoleon_links.to_csv('../hpc_jobs/hpc_spotlight/data/napoleon_links.csv', index=False)

## Save dataframe with sources ID's and titles

In [11]:
con = duckdb.connect('../data/out/napo_sources.db')
source_names = con.execute("""
    SELECT 'Q' || wikidata_id AS wikidata_id, page_title
    FROM wiki_page WHERE language_code = 'en'
""").df()

con.close()

source_names

,wikidata_id,page_title
0,Q63023,Friedrich Wilhelm Freiherr von Bülow
1,Q607190,Paolo Virzì
2,Q451608,Gaspard de Prony
3,Q334973,Félicité de La Mennais
4,Q115995,Achille Baraguey d'Hilliers
5,Q360481,"Vincent-Marie Viénot, Count of Vaublanc"
6,Q7243,Leo Tolstoy
7,Q57511,Franz Joseph Gall
8,Q78873,Ida Laura Pfeiffer
9,Q7721,Napoleon III


In [12]:
con = duckdb.connect('../data/out/napoleon_self.db')
napoleon_name = con.execute("""
    SELECT 'Q' || wikidata_id AS wikidata_id, page_title
    FROM wiki_page WHERE language_code = 'en'
""").df()

con.close()

napoleon_name

,wikidata_id,page_title
0,Q517,Napoleon


In [13]:
all_names = pd.concat(
    [napoleon_name, source_names],
    ignore_index=True
)
all_names

,wikidata_id,page_title
0,Q517,Napoleon
1,Q63023,Friedrich Wilhelm Freiherr von Bülow
2,Q607190,Paolo Virzì
3,Q451608,Gaspard de Prony
4,Q334973,Félicité de La Mennais
5,Q115995,Achille Baraguey d'Hilliers
6,Q360481,"Vincent-Marie Viénot, Count of Vaublanc"
7,Q7243,Leo Tolstoy
8,Q57511,Franz Joseph Gall
9,Q78873,Ida Laura Pfeiffer


In [ ]:
# Write file in HPC folder
all_names.to_csv('../hpc_jobs/hpc_spotlight/data/napoleon_source_names.csv', index=False)

## Relevant files for Napoleon's outgoing edges

In [15]:
con = duckdb.connect('../data/out/napoleon_self.db')

napoleon_biography = con.execute("""
    SELECT 'Q' || wp.wikidata_id, wp.language_code, e.extract
    FROM wiki_page wp
    JOIN extract e ON e.wiki_page_id = wp.wiki_page_id
""").fetchall()

con.close()

napoleon_biography = pd.DataFrame(napoleon_biography, columns=["wikidata_id", "language_code", "extract"])

print(napoleon_biography.shape)
napoleon_biography.head()

(1, 3)


,wikidata_id,language_code,extract
0,Q517,en,Napoleon Bonaparte (born Napoleone di Buonapar...


In [18]:
# Edges from Napoleon (Q517) to other people
napo_out = fullmatch_df[
    (fullmatch_df["source_wikidata_id"] == "Q517")
    & (fullmatch_df["language_code"] == "en")
]

napoleon_out_links = napo_out[["source_wikidata_id", "target_wikidata_id", "language_code"]]

print(napoleon_out_links.shape)
napoleon_out_links.head()

(403, 3)


,source_wikidata_id,target_wikidata_id,language_code
103760,Q517,Q708011,en
103761,Q517,Q8442,en
103762,Q517,Q152463,en
103763,Q517,Q157099,en
103764,Q517,Q208086,en


In [ ]:
# Write files in HPC folder
napoleon_biography.to_csv('../hpc_jobs/hpc_spotlight/data/napoleon_out/napoleon_biography.csv', index=False)
napoleon_out_links.to_csv('../hpc_jobs/hpc_spotlight/data/napoleon_out/napoleon_out_links.csv', index=False)

## Get ID's and titles for Napoleon's targets

**Run the following command in terminal to build the graph:**

```
python src/main.py build \
    --input data/napo_targets.csv \
    --output data/out/napo_targets.db \
    --languages en
```

In [20]:
con = duckdb.connect('../data/out/napo_targets.db')
target_names = con.execute("""
    SELECT 'Q' || wikidata_id AS wikidata_id, page_title
    FROM wiki_page WHERE language_code = 'en'
""").df()
con.close()

target_names

,wikidata_id,page_title
0,Q617241,François-Nicolas Vincent
1,Q242609,André Masséna
2,Q157521,Charles Lucien Bonaparte
3,Q151087,Jérôme Bonaparte
4,Q473506,"Louis Joseph, Dauphin of France"
...,...,...
398,Q152148,Wenceslaus IV of Bohemia
399,Q483405,Anacharsis Cloots
400,Q8404,John I of France
401,Q212068,Jean-Jacques Dessalines


In [ ]:
# Write file in HPC folder
target_names.to_csv('../hpc_jobs/hpc_relik/data/napoleon_target_names.csv', index=False)

# For mapping relik predictions to titles
target_names.to_csv('../hpc_jobs/hpc_relik_cie/data/napoleon_target_names.csv', index=False)
target_names.to_csv('../hpc_jobs/hpc_spotlight/data/napoleon_out/napoleon_target_names.csv', index=False)

# Used later to map LG and TRF predictions to titles
target_names.to_csv('../data/napoleon_target_names.csv', index=False)

## (Get Spotlight weights on HPC)

## Apply rest of Spotlight pipeline

### Incoming edges: source -> Napoleon

In [57]:
# Load spotlight predictions
in_weights_df = pd.read_csv('../data/weight_files/spotlight_weights.csv')

# QID -> title 
source_names = pd.read_csv('../data/napoleon_source_names.csv')   
titles = dict(zip(source_names["wikidata_id"], source_names["page_title"]))

In [58]:
print(in_weights_df.shape)
in_weights_df.head()

(46, 4)


,source_ID,target_ID,language_code,spotlight_weight
0,Q63023,Q517,en,4
1,Q607190,Q517,en,3
2,Q451608,Q517,en,1
3,Q334973,Q517,en,6
4,Q115995,Q517,en,1


In [59]:
# Napoleon III
in_weights_df[in_weights_df['source_ID']=='Q7721']

,source_ID,target_ID,language_code,spotlight_weight
9,Q7721,Q517,en,309


In [60]:
# Set weight to 1 if the target title is a substring of the source title
def is_substring(row):
    src = titles.get(row["source_ID"])
    trg = titles.get(row["target_ID"])
    if not src or not trg:
        return False
    return f" {trg.lower()} " in f" {src.lower()} "

in_weights_df.loc[in_weights_df.apply(is_substring, axis=1), "spotlight_weight"] = 1

In [61]:
# Napoleon III after substring step
in_weights_df[in_weights_df['source_ID']=='Q7721']

,source_ID,target_ID,language_code,spotlight_weight
9,Q7721,Q517,en,1


In [62]:
# Save corrected weights
in_weights_df.to_csv('../data/weight_files/spotlight_weights_corrected.csv', index=False)

### Outgoing edges: Napoleon -> target

In [63]:
# Load spotlight predictions
out_weights_df = pd.read_csv('../data/weight_files/napoleon_out_edges/spotlight_napoleon_weights.csv')

In [64]:
print(out_weights_df.shape)
out_weights_df.head()

(403, 4)


,source_ID,target_ID,language_code,spotlight_weight
0,Q517,Q708011,en,0
1,Q517,Q8442,en,0
2,Q517,Q152463,en,0
3,Q517,Q157099,en,0
4,Q517,Q208086,en,0


In [65]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
from src.edge_processing import count_exact_matches_grouped
from src.filters import normalize_title, DEFAULT_STOPWORDS

# Same text and titles Spotlight used
napoleon_biography = pd.read_csv('../hpc_jobs/hpc_spotlight/data/napoleon_out/napoleon_biography.csv')
target_names = pd.read_csv('../data/napoleon_target_names.csv')
source_names = pd.read_csv('../data/napoleon_source_names.csv')

titles = dict(zip(pd.concat([target_names, source_names])["wikidata_id"],
                  pd.concat([target_names, source_names])["page_title"]))

In [66]:
# has_shared_word (from has_shared_word_func in src/edge_processing.py)
def name_words(title):
    return [w for w in normalize_title(str(title)).lower().split()
            if w not in DEFAULT_STOPWORDS and len(w) > 3]

def has_shared_word(row):
    src, trg = titles.get(row['source_ID']), titles.get(row['target_ID'])
    if not src or not trg:
        return False
    return any(w in name_words(src) for w in name_words(trg))

out_weights_df["has_shared_word"] = out_weights_df.apply(has_shared_word, axis=1)

In [67]:
out_weights_df.head()

,source_ID,target_ID,language_code,spotlight_weight,has_shared_word
0,Q517,Q708011,en,0,False
1,Q517,Q8442,en,0,False
2,Q517,Q152463,en,0,False
3,Q517,Q157099,en,0,False
4,Q517,Q208086,en,0,False


In [68]:
# Compute z-score for Napoleons outgoing weights
# z-score (from compute_z_score: per source, zeros included, np.std = ddof=0)
w = out_weights_df["spotlight_weight"]
out_weights_df["z_score"] = (w - w.mean()) / w.std(ddof=0)

In [69]:
# full-name count
edges_tmp = out_weights_df.rename(columns={'source_ID': "source_wikidata_id", 'target_ID': "target_wikidata_id"})

src_extracts = {(r.wikidata_id, r.language_code): r.extract for r in napoleon_biography.itertuples()}
tgt_titles   = {(r.wikidata_id, "en"): r.page_title for r in target_names.itertuples()}

out_weights_df["fullmatch_count"] = count_exact_matches_grouped(edges_tmp, src_extracts, tgt_titles)

Grouped 403 edges into 1 source groups (avg 403.0 targets/source).


Source groups (exact): 100%|██████████| 1/1 [00:00<00:00, 66.94it/s]


In [70]:
flag = out_weights_df["has_shared_word"] & (out_weights_df["z_score"] > 1.5)

# Look at shared-name edges before correcting
out_weights_df[out_weights_df["has_shared_word"]][['target_ID', 'spotlight_weight', "z_score", "fullmatch_count"]]

,target_ID,spotlight_weight,z_score,fullmatch_count
25,Q7721,0,-0.286349,2
171,Q7723,0,-0.286349,4


In [71]:
# Step 1 from the paper: weight = full-name count for flagged edges
out_weights_df.loc[flag, 'spotlight_weight'] = out_weights_df.loc[flag, "fullmatch_count"]
print(f"Corrected {flag.sum()} edges")

Corrected 0 edges
